<a href="https://colab.research.google.com/github/pierre-roth/mlfcs-gapa/blob/mm-drl-lob/paper_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# initial try, but not working well here due to low data volume. keeping it for later as then Attn-LOB will be useful
# for right now, use the code in the second cell with SimpleLOB)
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

CFG = dict(
    ask_path="/content/ask.csv",       # ask1..10 price/vol
    bid_path="/content/bid.csv",       # bid1..10 price/vol
    msg_path="/content/msg.csv",       # message flow
    bbo_path="/content/price.csv",     # ask1_price, bid1_price, midprice
    trades_path="/content/trades.csv", # price, size, aggressor_side

    n_levels=10,
    T=50,
    tick=0.01,

    # pretrain
    k=100,
    alpha=1e-5,
    pretrain_epochs=10,
    pretrain_lr=1e-3,
    batch_size=128,

    # env
    episode_len=2000,
    min_trade_steps=12,   # NEW: reject weak episodes
    max_inv=10,
    trade_unit=10,
    max_bias=0.1,
    max_spread=0.2,
    eta=0.5,
    zeta=1e-4,
    latency=1,

    # PPO
    rl_epochs=50,         # was 20
    eps_per_epoch=32,     # was 8
    rl_lr=3e-4,
    gamma=0.99,
    lam=0.95,
    ppo_clip=0.2,
    ppo_updates=4,
    mb_size=64,           # was 256; better for your rollout size
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ── 1. Data ──────────────────────────────────────────────────────────────────

def load_data(cfg):
    ask = pd.read_csv(cfg["ask_path"])
    bid = pd.read_csv(cfg["bid_path"])
    msg = pd.read_csv(cfg["msg_path"])
    bbo = pd.read_csv(cfg["bbo_path"])
    n = min(len(ask), len(bid), len(msg), len(bbo))

    d = pd.DataFrame({"timestamp": pd.to_datetime(ask["timestamp"].iloc[:n])})
    for i in range(1, cfg["n_levels"] + 1):
        d[f"ask{i}_price"]  = ask[f"ask{i}_price"].iloc[:n].values
        d[f"ask{i}_volume"] = ask[f"ask{i}_volume"].iloc[:n].values
        d[f"bid{i}_price"] = bid[f"bid{i}_price"].iloc[:n].values
        d[f"bid{i}_volume"] = bid[f"bid{i}_volume"].iloc[:n].values
    d["midprice"]  = bbo["midprice"].iloc[:n].values
    d["best_ask"]  = bbo["ask1_price"].iloc[:n].values
    d["best_bid"]  = bbo["bid1_price"].iloc[:n].values
    d["spread"]    = d["best_ask"] - d["best_bid"]
    for c in ["market_buy_volume","market_sell_volume","limit_buy_volume",
              "limit_sell_volume","withdraw_buy_volume","withdraw_sell_volume",
              "market_buy_n","market_sell_n","limit_buy_n","limit_sell_n",
              "withdraw_buy_n","withdraw_sell_n"]:
        d[c] = msg[c].iloc[:n].values
    return d


def load_trades(cfg):
    tr = pd.read_csv(cfg["trades_path"])
    tr["timestamp"] = pd.to_datetime(tr["timestamp"])
    tr = tr.sort_values("timestamp").reset_index(drop=True)
    return tr


def build_lob_array(data, nl=10):
    """Build raw (N, 40) array: ask1_p, ask1_v, bid1_p, bid1_v, ..."""
    cols = []
    for i in range(1, nl + 1):
        cols += [f"ask{i}_price", f"ask{i}_volume", f"bid{i}_price", f"bid{i}_volume"]
    return data[cols].values.astype(np.float32)


# Normalization
# Do this such that prices become stationary looking, because they are expressed as percentages around the current mid instead of raw absolute prices.
# Not 1-to-1 as paper, but the volumes are normalized using log1p + z-norm (rather than volume/max)

def fit_lob_normalizer(raw_lob, nl=10):
    x = raw_lob.copy().astype(np.float32)
    mid = (x[:, 0] + x[:, 2]) / 2 + 1e-8
    for i in range(nl):
        x[:, 4*i]   = x[:, 4*i]   / mid - 1.0
        x[:, 4*i+2] = x[:, 4*i+2] / mid - 1.0
    for i in range(nl):
        x[:, 4*i+1] = np.log1p(x[:, 4*i+1])
        x[:, 4*i+3] = np.log1p(x[:, 4*i+3])
    mean = x.mean(axis=0).astype(np.float32)
    std = (x.std(axis=0) + 1e-8).astype(np.float32)
    return mean, std

def apply_lob_normalizer(raw_lob, nl, mean, std):
    x = raw_lob.copy().astype(np.float32)
    mid = (x[:, 0] + x[:, 2]) / 2 + 1e-8
    for i in range(nl):
        x[:, 4*i]   = x[:, 4*i]   / mid - 1.0
        x[:, 4*i+2] = x[:, 4*i+2] / mid - 1.0
    for i in range(nl):
        x[:, 4*i+1] = np.log1p(x[:, 4*i+1])
        x[:, 4*i+3] = np.log1p(x[:, 4*i+3])
    return ((x - mean) / std).astype(np.float32)

# Labels
# Creates the pretraining target -> turns the raw midprice series into a supervised learning problem:
# “Given the last 50 LOB states, predict whether price will go down / stay / go up.”

def make_labels(midprice, k=10, alpha=1e-5):
    """Author's getLabel: rolling mean past vs shifted rolling mean future."""
    mp = pd.Series(midprice)
    price_past = mp.rolling(window=k).mean()
    price_future = mp.copy()
    price_future.iloc[:-k] = price_past.iloc[k:].values
    price_future.iloc[-k:] = np.nan
    pct = (price_future - price_past) / (price_past + 1e-12)
    y = np.full(len(mp), -1, dtype=np.int64)
    y[pct >= alpha] = 2   # up
    y[pct <= -alpha] = 0  # down
    y[(pct > -alpha) & (pct < alpha)] = 1  # stationary
    return y

# Attn-LOB (matching author's network.py EXACTLY) ──────────────────────

class AttnLOB(nn.Module):
    """
    Author's architecture is SEQUENTIAL not parallel (I first programmed it parallel):
      Conv2d(1,32,(1,2),s=(1,2)) → 2x Conv2d(32,32,(4,1)) →
      Conv2d(32,32,(1,5),s=(1,5)) → 2x Conv2d(32,32,(4,1)) →
      Conv2d(32,32,(1,4))         → 2x Conv2d(32,32,(4,1)) →
      Inception(3 branches) → Reshape → CrossAttention(query=last, kv=all) → output 64-dim
    """
    def __init__(self, T=50):
        super().__init__()
        self.T = T
        self.feat_dim = 64  # author's output_shape=64 in MultiHeadAttention

        # Sequential spatial convs (DeepLOB-style)
        self.spatial = nn.Sequential(
            nn.Conv2d(1, 32, (1, 2), stride=(1, 2)), nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, (4, 1), padding=(1, 0)), nn.LeakyReLU(0.01),  # padding='same' approx
            nn.Conv2d(32, 32, (4, 1), padding=(1, 0)), nn.LeakyReLU(0.01),

            nn.Conv2d(32, 32, (1, 5), stride=(1, 5)), nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, (4, 1), padding=(1, 0)), nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, (4, 1), padding=(1, 0)), nn.LeakyReLU(0.01),

            nn.Conv2d(32, 32, (1, 4)), nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, (4, 1), padding=(1, 0)), nn.LeakyReLU(0.01),
            nn.Conv2d(32, 32, (4, 1), padding=(1, 0)), nn.LeakyReLU(0.01),
        )

        # Inception module (author: 3 branches, no 1x1 standalone)
        self.inc_31 = nn.Sequential(nn.Conv2d(32, 64, (1, 1)), nn.LeakyReLU(0.01),
                                    nn.Conv2d(64, 64, (3, 1), padding=(1, 0)), nn.LeakyReLU(0.01))
        self.inc_51 = nn.Sequential(nn.Conv2d(32, 64, (1, 1)), nn.LeakyReLU(0.01),
                                    nn.Conv2d(64, 64, (5, 1), padding=(2, 0)), nn.LeakyReLU(0.01))
        self.inc_mp = nn.Sequential(nn.MaxPool2d((3, 1), stride=(1, 1), padding=(1, 0)),
                                    nn.Conv2d(32, 64, (1, 1)), nn.LeakyReLU(0.01))

        # Cross-attention: query = last timestep, key/value = all timesteps
        # Author: num_heads=10, key_dim=16, output_shape=64
        self.attn = nn.MultiheadAttention(embed_dim=192, num_heads=8, batch_first=True)
        self.attn_proj = nn.Linear(192, 64)

        # Pretrain head
        self.cls = nn.Linear(64, 3)

    def features(self, x):
        # x: (B, T, 40) where B is batch size, T is lookback length and 40 raw LOB features
        B = x.shape[0]
        x = x.unsqueeze(1)  # (B, 1, T, 40)
        h = self.spatial(x)  # (B, 32, T', 1) — T' may differ from T due to conv

        # inception
        i1 = self.inc_31(h)
        i2 = self.inc_51(h)
        i3 = self.inc_mp(h)
        h = torch.cat([i1, i2, i3], dim=1)  # (B, 192, T', 1)

        # reshape to (B, T', 192)
        h = h.squeeze(-1).permute(0, 2, 1)

        # cross-attention: query = last timestep only
        query = h[:, -1:, :]  # (B, 1, 192)
        out, _ = self.attn(query, h, h)  # (B, 1, 192)
        out = out.squeeze(1)  # (B, 192)
        out = self.attn_proj(out)  # (B, 64)
        return out

    def forward(self, x):
        return self.cls(self.features(x))


# ── 3. Pretrain ──────────────────────────────────────────────────────────────

class LOBDataset(Dataset):
    def __init__(self, raw_lob, labels, T, k, mean, std, nl=10):
        self.raw, self.labels, self.T = raw_lob, labels, T
        self.mean, self.std, self.nl = mean, std, nl
        self.idx = np.where((np.arange(len(labels)) >= T) &
                            (np.arange(len(labels)) < len(labels)-k) &
                            (labels >= 0))[0]

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        t = self.idx[i]
        window = self.raw[t-self.T:t]
        normed = apply_lob_normalizer(window, self.nl, self.mean, self.std)
        return (torch.tensor(normed, dtype=torch.float32),
                torch.tensor(self.labels[t], dtype=torch.long))


def pretrain(data, raw_lob, lob_mean, lob_std, cfg):
    print("── Pretrain Attn-LOB ──")
    labels = make_labels(data["midprice"].values, cfg["k"], cfg["alpha"])
    nl = cfg["n_levels"]
    n_tr = int(len(raw_lob) * 0.8)

    tr = DataLoader(LOBDataset(raw_lob[:n_tr], labels[:n_tr], cfg["T"], cfg["k"], lob_mean, lob_std, nl),
                    cfg["batch_size"], shuffle=True, drop_last=True)
    va = DataLoader(LOBDataset(raw_lob[n_tr:], labels[n_tr:], cfg["T"], cfg["k"], lob_mean, lob_std, nl),
                    cfg["batch_size"])

    model = AttnLOB(cfg["T"]).to(DEVICE)
    opt = optim.Adam(model.parameters(), cfg["pretrain_lr"], weight_decay=1e-5)
    cnt = np.bincount(labels[:n_tr][labels[:n_tr] >= 0], minlength=3).astype(float) + 1
    w = torch.tensor(3.0 / cnt / cnt.sum(), dtype=torch.float32).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)
    best, best_sd = 0, None

    for ep in range(cfg["pretrain_epochs"]):
        model.train()
        for xb, yb in tr:
            loss = crit(model(xb.to(DEVICE)), yb.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); preds, trues = [], []
        with torch.no_grad():
            for xb, yb in va:
                preds.append(model(xb.to(DEVICE)).argmax(1).cpu())
                trues.append(yb)
        p, t = torch.cat(preds).numpy(), torch.cat(trues).numpy()
        f1s = []
        for c in range(3):
            tp = ((p==c)&(t==c)).sum(); fp = ((p==c)&(t!=c)).sum(); fn = ((p!=c)&(t==c)).sum()
            pr = tp/(tp+fp+1e-8); rc = tp/(tp+fn+1e-8)
            f1s.append(2*pr*rc/(pr+rc+1e-8))
        f1 = np.mean(f1s)
        if f1 > best:
            best = f1; best_sd = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if ep % 5 == 0 or ep == cfg["pretrain_epochs"]-1:
            print(f"  ep {ep:3d}  acc={(p==t).mean():.3f}  F1={f1:.3f}")

    model.load_state_dict(best_sd)
    print(f"  Best F1={best:.3f}")
    return model


# ── 4. Dynamic features (author's env_feature.py) ───────────────────────────

DYN_DIM = 24   # 6 market state + 18 OSI

def dynamic_features(data, idx, ts):
    """6 = RV×3 + RSI×3, then 18 = OSI×3 windows × 6 each. Total 24."""
    f = []
    # RV and RSI at 300s, 600s, 1800s (author's _get_market_state)
    mp = data["midprice"].values[:idx+1]
    for wsec in [5, 15, 60]:
        mask = ts[:idx+1] >= ts[idx] - wsec
        w = mp[max(0, idx+1-mask.sum()):]
        if len(w) > 1:
            lr = np.diff(np.log(w + 1e-12))
            f.append(np.sqrt((lr**2).sum()) * 1e4)  # RV, scaled like author
        else:
            f.append(0.0)
    for wsec in [5, 15, 60]:
        mask = ts[:idx+1] >= ts[idx] - wsec
        w = mp[max(0, idx+1-mask.sum()):]
        if len(w) > 1:
            d = np.diff(w)
            g, l = np.maximum(d, 0).sum(), np.maximum(-d, 0).sum()
            f.append(g / (g + l + 1e-8))
        else:
            f.append(0.5)

    # OSI at 10s, 60s, 300s (author's _get_order_strength_index)
    for wsec in [10, 60, 300]:
        mask = ts[:idx+1] >= ts[idx] - wsec
        w = data.iloc[max(0, idx+1-mask.sum()):idx+1]
        if len(w):
            mbv, msv = w["market_buy_volume"].sum(), w["market_sell_volume"].sum()
            mbn, msn = w["market_buy_n"].sum(), w["market_sell_n"].sum()
            lbv, lsv = w["limit_buy_volume"].sum(), w["limit_sell_volume"].sum()
            lbn, lsn = w["limit_buy_n"].sum(), w["limit_sell_n"].sum()
            cbv, csv = w["withdraw_buy_volume"].sum(), w["withdraw_sell_volume"].sum()
            cbn, csn = w["withdraw_buy_n"].sum(), w["withdraw_sell_n"].sum()
            f += [(mbv-msv)/(mbv+msv+1e-7), (mbn-msn)/(mbn+msn+1e-7),
                  (lbv-lsv)/(lbv+lsv+1e-7), (lbn-lsn)/(lbn+lsn+1e-7),
                  (cbv-csv)/(cbv+csv+1e-7), (cbn-csn)/(cbn+csn+1e-7)]
        else:
            f += [0]*6
    return np.array(f, dtype=np.float32)


# ── 5. RL Environment (matching author's base_env.py + env_continuous.py) ───────

import math

def price_legal_check(ask, bid, tick=0.01):
    """Author's utils.py: ceil ask, floor bid to tick."""
    ask = math.ceil(ask / tick) * tick
    bid = math.floor(bid / tick) * tick
    return round(ask, 2), round(bid, 2)


class MMEnv:
    def __init__(self, data, raw_lob, trades, cfg, lob_mean, lob_std):
        self.data, self.raw_lob, self.cfg = data, raw_lob, cfg
        self.lob_mean, self.lob_std = lob_mean, lob_std
        self.ts = (data["timestamp"] - data["timestamp"].iloc[0]).dt.total_seconds().values

        self.is_trade = np.zeros(len(data), dtype=bool)
        # Pre-index trades by timestamp for O(1) lookup in _match
        self.trades_by_ts = {}
        if trades is not None and len(trades):
            for ts_val, grp in trades.groupby("timestamp"):
                self.trades_by_ts[ts_val] = grp
            trade_ts_set = set(self.trades_by_ts.keys())
            for i, t in enumerate(data["timestamp"].values):
                if t in trade_ts_set:
                    self.is_trade[i] = True
        self.trades_df = trades

    def reset(self, start=None):
      c = self.cfg
      T = c["T"]
      min_trade_steps = c.get("min_trade_steps", 12)

      mx = len(self.data) - c["episode_len"] - T - 1
      mx = max(T + 1, mx)

      trade_idx = None

      # Try several times to find an episode with enough trade decisions
      for _ in range(100):
        self.ep_start = start if start is not None else np.random.randint(T, mx)
        self.ep_end = min(self.ep_start + c["episode_len"], len(self.data))

        ep_is_trade = self.is_trade[self.ep_start:self.ep_end]
        trade_idx = np.where(ep_is_trade)[0]
        trade_idx = trade_idx[trade_idx > T]  # need T lookback

        if len(trade_idx) >= min_trade_steps:
            break

    # Fallback: use whatever was found, even if weak
      if trade_idx is None or len(trade_idx) == 0:
        trade_idx = np.array([T], dtype=int)

      self.trade_indices = iter(trade_idx)

      self.inv = 0
      self.cash = 0.0
      self.value = 0.0
      self.prev_value = 0.0
      self.n_trades = 0
      self.vol = 0
      self.inv_hist = []
      self.mid_price_prev = None

      try:
        self.i = next(self.trade_indices)
        self.i_next = next(self.trade_indices)
      except StopIteration:
        self.i = T
        self.i_next = None

      return self._obs()

    def _abs_idx(self): return self.ep_start + self.i


    def _get_price(self, rel_idx):
        row = self.data.iloc[self.ep_start + rel_idx]
        ba = row["best_ask"]; bb = row["best_bid"]
        return (ba + bb) / 2, ba, bb, ba - bb


    def _obs(self):
        ai = self._abs_idx()
        c = self.cfg; T = c["T"]; lat = c["latency"]
        obs_i = max(0, ai - lat)

        raw = self.raw_lob[max(0, obs_i-T):obs_i]
        if len(raw) < T:
            raw = np.vstack([np.zeros((T-len(raw), raw.shape[1]), np.float32), raw])
        lob = apply_lob_normalizer(raw, c["n_levels"], self.lob_mean, self.lob_std)

        dyn = dynamic_features(self.data, obs_i, self.ts)

        inv_frac = self.inv / (c["max_inv"] * c["trade_unit"])
        time_frac = self.i / c["episode_len"]
        ag = np.array([inv_frac]*12 + [time_frac]*12, dtype=np.float32)
        self._last_obs = (lob, dyn, ag)  # cache
        return lob, dyn, ag

    def step(self, a1, a2):
        c = self.cfg; tu = c["trade_unit"]; tick = c["tick"]; lat = c["latency"]

        mid, ba, bb, spread = self._get_price(self.i)
        if self.mid_price_prev is None:
            self.mid_price_prev = mid

        mid_lat, a1_lat, b1_lat, _ = self._get_price(self.i - lat)
        delta = a1 * c["max_bias"]
        if self.inv > 0:
            pr = mid_lat - delta
        elif self.inv < 0:
            pr = mid_lat + delta
        else:
            pr = mid_lat
        sp = a2 * c["max_spread"]
        ask_p = pr + sp / 2
        bid_p = pr - sp / 2
        ask_p, bid_p = price_legal_check(ask_p, bid_p, tick)

        if self.inv <= -c["max_inv"] * tu:
            ask_p = 0
        elif self.inv >= c["max_inv"] * tu:
            bid_p = 0

        trade_price, trade_volume = self._match(ask_p, bid_p, tu)

        self.inv += trade_volume
        self.cash -= trade_volume * trade_price
        self.value = self.cash + self.inv * mid

        pnl = self.value - self.prev_value

        # Paper Eq. 16: R_t = DP_t + TP_t - IP_t
        # Dampened PnL: penalises profit from holding but not losses
        #dp = pnl - max(0.0, c["eta"] * pnl)
        # Trading PnL: reward price advantage relative to mid
        #tp = trade_volume * (mid - trade_price) if trade_volume else 0.0
        # Inventory Punishment: L2 penalty on position
        #ip = c["zeta"] * self.inv ** 2

        # Paper-style reward, but scaled sanely for equity-share data
        dp = pnl - max(0.0, c["eta"] * pnl)
        tp = trade_volume * (mid - trade_price) if trade_volume else 0.0
        inv_lots = self.inv / c["trade_unit"]
        ip = c["zeta"] * (inv_lots ** 2)

        reward = dp + tp - ip
        self.prev_value = self.value
        self.mid_price_prev = mid

        if trade_volume != 0:
            self.n_trades += 1
            self.vol += abs(trade_volume * trade_price)

        self.inv_hist.append(self.inv)

        # save last valid index before advancing
        prev_i = self.i

        terminal = False
        if self.i_next is None:
          terminal = True
        else:
          self.i = self.i_next
        try:
          self.i_next = next(self.trade_indices)
        except StopIteration:
          self.i_next = None

        if terminal and self.inv:
            # use prev_i for price lookup since self.i may be invalid
            _, a1_close, b1_close, _ = self._get_price(prev_i)
            if self.inv > 0:
                tv = -self.inv
                tp = b1_close
            else:
                tv = -self.inv
                tp = a1_close
            self.inv += tv
            self.cash -= tv * tp
            self.value = self.cash + self.inv * mid
            reward += self.value - self.prev_value
            self.prev_value = self.value

        obs = self._obs() if not terminal else self._last_obs
        return obs, reward, terminal, {
            "val": self.value, "trades": self.n_trades, "inv": self.inv
        }

    def _match(self, ask_p, bid_p, tu):
        ai = self._abs_idx()
        ts = self.data.iloc[ai]["timestamp"]

        now_trades = self.trades_by_ts.get(ts)
        if now_trades is None or len(now_trades) == 0:
            return 0, 0

        buy_trades = now_trades[now_trades["aggressor_side"] == "B"] # buy MOs lift asks
        sell_trades = now_trades[now_trades["aggressor_side"] == "A"] # sell MOs hit bids

        _, a1_prev, b1_prev, _ = self._get_price(self.i - 1)

        trade_price, trade_volume = 0, 0

        # our ask fills against buy-aggressive trades
        if ask_p and ask_p > 0 and len(buy_trades):
            max_tp = buy_trades["price"].max()
            max_tv = buy_trades[buy_trades["price"] == max_tp]["size"].sum()
            if ask_p <= b1_prev:
                trade_price, trade_volume = b1_prev, -tu
            elif max_tp > ask_p:
                trade_price, trade_volume = ask_p, -tu
            elif max_tp == ask_p:
                depth = self.data.iloc[ai].get("ask1_volume", 100)
                prob = max_tv / (max_tv + depth + 1e-7)
                if np.random.random() < prob:
                    trade_price, trade_volume = ask_p, -tu

        # our bid fills against sell-aggressive trades
        if bid_p and bid_p > 0 and len(sell_trades):
            min_tp = sell_trades["price"].min()
            min_tv = sell_trades[sell_trades["price"] == min_tp]["size"].sum()
            if bid_p >= a1_prev:
                trade_price, trade_volume = a1_prev, tu
            elif min_tp < bid_p:
                trade_price, trade_volume = bid_p, tu
            elif min_tp == bid_p:
                depth = self.data.iloc[ai].get("bid1_volume", 100)
                prob = min_tv / (min_tv + depth + 1e-7)
                if np.random.random() < prob:
                    trade_price, trade_volume = bid_p, tu

        return trade_price, trade_volume


# ── PPO Policy ────────────────────────────────────────────────────────────
class Policy(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.bb = backbone

        # UNFREEZE backbone so PPO can adapt the LOB representation
        for p in self.bb.parameters():
            p.requires_grad = True

        feat = backbone.feat_dim

        self.trunk = nn.Sequential(
            nn.Linear(feat + DYN_DIM + 24, 64),
            nn.LeakyReLU(0.01),
        )

        # Beta parameters for 2 bounded actions
        self.alpha_head = nn.Linear(64, 2)
        self.beta_head  = nn.Linear(64, 2)

        self.val = nn.Linear(64, 1)

    def forward(self, lob, dyn, ag):
        # IMPORTANT: no torch.no_grad() here
        f = self.bb.features(lob)
        h = self.trunk(torch.cat([f, dyn, ag], dim=-1))

        alpha = F.softplus(self.alpha_head(h)) + 1.0
        beta  = F.softplus(self.beta_head(h)) + 1.0
        value = self.val(h).squeeze(-1)
        return alpha, beta, value

    def act(self, lob, dyn, ag, det=False):
        alpha, beta, v = self(lob, dyn, ag)
        d = torch.distributions.Beta(alpha, beta)

        if det:
            a = alpha / (alpha + beta)
            logp = torch.zeros(a.shape[0], device=a.device)
            return a, logp, v

        a = d.rsample()
        logp = d.log_prob(a).sum(-1)
        return a, logp, v

# ── 7. Train ─────────────────────────────────────────────────────────────────

def gae(rews, vals, dones, gamma, lam):
    T = len(rews); adv = np.zeros(T, np.float32); g = nv = 0.0
    for t in reversed(range(T)):
        if dones[t]: nv = g = 0.0
        g = rews[t] + gamma*nv - vals[t] + gamma*lam*g
        adv[t] = g; nv = vals[t]
    return adv, adv + np.array(vals)

def to_t(x): return torch.tensor(x, dtype=torch.float32, device=DEVICE)

def train(env, policy, cfg):
    print(f"\n── PPO ({cfg['rl_epochs']} ep × {cfg['eps_per_epoch']} traj) ──")
    opt = optim.Adam(filter(lambda p: p.requires_grad, policy.parameters()), cfg["rl_lr"])

    for epoch in range(cfg["rl_epochs"]):
        obs_l, obs_d, obs_a, acts, logps, rews, vals, dns = [],[],[],[],[],[],[],[]
        ep_r, ep_tr = [], []
        policy.eval()
        for _ in range(cfg["eps_per_epoch"]):
            lob, dyn, ag = env.reset(); er = 0
            while True:
                lt, dt, at = to_t(lob).unsqueeze(0), to_t(dyn).unsqueeze(0), to_t(ag).unsqueeze(0)
                with torch.no_grad(): act, lp, v = policy.act(lt, dt, at)
                an = act.squeeze(0).cpu().numpy()
                (lob2, dyn2, ag2), r, done, info = env.step(an[0], an[1])
                obs_l.append(lob); obs_d.append(dyn); obs_a.append(ag)
                acts.append(an); logps.append(lp.item())
                rews.append(r); vals.append(v.item()); dns.append(done)
                er += r; lob, dyn, ag = lob2, dyn2, ag2
                if done: break
            ep_r.append(er); ep_tr.append(info["trades"])

        if not rews: continue
        adv, ret = gae(rews, vals, dns, cfg["gamma"], cfg["lam"])
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        policy.train(); N = len(rews); idx = np.arange(N)
        Tl = to_t(np.stack(obs_l)); Td = to_t(np.stack(obs_d)); Ta = to_t(np.stack(obs_a))
        Tact = to_t(np.stack(acts)); Tolp = to_t(logps); Tadv = to_t(adv); Tret = to_t(ret)

        for _ in range(cfg["ppo_updates"]):
            np.random.shuffle(idx)
            for s in range(0, N, cfg["mb_size"]):
                mb = idx[s:s+cfg["mb_size"]]

                alpha, beta, v = policy(Tl[mb], Td[mb], Ta[mb])
                dist = torch.distributions.Beta(alpha, beta)

                # avoid exact 0 or 1 for numerical stability
                acts_mb = Tact[mb].clamp(1e-6, 1 - 1e-6)

                nlp = dist.log_prob(acts_mb).sum(-1)
                ratio = (nlp - Tolp[mb]).exp()
                cl = ratio.clamp(1-cfg["ppo_clip"], 1+cfg["ppo_clip"])
                loss = -torch.min(ratio*Tadv[mb], cl*Tadv[mb]).mean() + 0.5*F.mse_loss(v, Tret[mb])
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), 0.5); opt.step()

        if epoch % 10 == 0 or epoch == cfg["rl_epochs"]-1:
            print(f"  ep {epoch:4d}  rew={np.mean(ep_r):+8.2f}  trades={np.mean(ep_tr):5.1f}")
    return policy


In [ ]:
# Run
if __name__ == "__main__":
    data = load_data(CFG)
    trades = load_trades(CFG)
    raw_lob = build_lob_array(data, CFG["n_levels"])

    # fit normalizer on training portion only
    n_tr = int(len(raw_lob) * 0.8)
    lob_mean, lob_std = fit_lob_normalizer(raw_lob[:n_tr], CFG["n_levels"])
    print(f"LOB: {len(data)} events, Trades: {len(trades)} fills, device={DEVICE}")

    backbone = pretrain(data, raw_lob, lob_mean, lob_std, CFG)
    env = MMEnv(data, raw_lob, trades, CFG, lob_mean, lob_std)
    policy = Policy(backbone).to(DEVICE)
    print(f"Trainable: {sum(p.numel() for p in policy.parameters() if p.requires_grad):,} params")

    policy = train(env, policy, CFG)
    torch.save(policy.state_dict(), "mm_policy.pt")
    print("Saved mm_policy.pt")


LOB: 49736 events, Trades: 1061 fills, device=cuda
── Pretrain Attn-LOB ──
  ep   0  acc=0.458  F1=0.209
  ep   5  acc=0.279  F1=0.146
  ep   9  acc=0.458  F1=0.209
  Best F1=0.209
Trainable: 241,704 params

── PPO (50 ep × 32 traj) ──
  ep    0  rew=   -0.05  trades=  0.5
  ep   10  rew=   +0.00  trades=  0.0
  ep   20  rew=   +0.00  trades=  0.0
  ep   30  rew=   +0.00  trades=  0.0
  ep   40  rew=   +0.00  trades=  0.1
  ep   49  rew=   +0.00  trades=  0.0
Saved mm_policy.pt


In [ ]:
"""
Market-Making RL Agent — v3 (Complete Rewrite)
================================================
Previous attempts failed due to TWO independent bugs:

PRETRAIN BUG: AttnLOB has 241K params but only ~4K effective training samples.
  Deep CNN-attention gets stuck outputting a single class (loss flat at 1.099).
  FIX: SimpleLOB with 10K params — 24× smaller. Uses 1D convolutions that can
  actually learn from limited data.

RL BUG: max_spread=0.20 means agent quotes ±$0.05 from mid.
  But AAPL buy trades only reach mid+$0.015 at the 99th percentile.
  Agent NEVER gets filled → 0 trades per episode.
  FIX: max_spread=0.03 (quotes at ±$0.0075 initially → fills possible).

Run:
    data = load_data(CFG)
    trades = load_trades(CFG)
    raw_lob = build_lob_array(data)
    lob_mean, lob_std = fit_lob_normalizer(raw_lob)
    model = pretrain(data, raw_lob, lob_mean, lob_std, CFG)
    policy = Policy(model).to(DEVICE)
    env = MMEnv(data, raw_lob, trades, CFG, lob_mean, lob_std)
    policy = train(env, policy, CFG)
"""

import numpy as np
import pandas as pd
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

CFG = dict(
    ask_path="/content/ask.csv",
    bid_path="/content/bid.csv",
    msg_path="/content/msg.csv",
    bbo_path="/content/price.csv",
    trades_path="/content/trades.csv",

    n_levels=10,
    T=50,
    tick=0.01,
    k=100,                # ~600ms horizon at 6ms/event (matches paper's ~1s)
    alpha=1e-5,

    # Pretrain
    pretrain_epochs=60,
    pretrain_lr=1e-3,
    batch_size=128,

    # RL env
    episode_len=2000,
    max_inv=10,
    trade_unit=10,

    # CRITICAL: calibrated for AAPL $265 with $0.02 spread
    # Buy trades reach mid+$0.015 at 99th pctl, mid+$0.01 at 90th pctl
    max_bias=0.02,         # before => max skew = 1 tick ($0.01)
    max_spread=0.06,       # initial quotes at ±$0.0075 → fills at ~70th pctl

    eta=0.5,
    zeta=1e-4,
    latency=1,

    # RL training
    rl_epochs=150,
    eps_per_epoch=32,
    rl_lr=3e-4,
    gamma=0.99,
    lam=0.95,
    ppo_clip=0.2,
    ppo_updates=4,
    mb_size=256,

    # Dynamic feature windows for 5-min dataset
    rv_rsi_windows=[5, 15, 60],
    osi_windows=[2, 10, 30],
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════════════════════
# 1. DATA
# ═══════════════════════════════════════════════════════════════════════════

def load_data(cfg):
    ask = pd.read_csv(cfg["ask_path"])
    bid = pd.read_csv(cfg["bid_path"])
    msg = pd.read_csv(cfg["msg_path"])
    bbo = pd.read_csv(cfg["bbo_path"])
    n = min(len(ask), len(bid), len(msg), len(bbo))
    d = pd.DataFrame({"timestamp": pd.to_datetime(ask["timestamp"].iloc[:n])})
    for i in range(1, cfg["n_levels"] + 1):
        d[f"ask{i}_price"] = ask[f"ask{i}_price"].iloc[:n].values
        d[f"ask{i}_volume"] = ask[f"ask{i}_volume"].iloc[:n].values
        d[f"bid{i}_price"] = bid[f"bid{i}_price"].iloc[:n].values
        d[f"bid{i}_volume"] = bid[f"bid{i}_volume"].iloc[:n].values
    d["midprice"] = bbo["midprice"].iloc[:n].values
    d["best_ask"] = bbo["ask1_price"].iloc[:n].values
    d["best_bid"] = bbo["bid1_price"].iloc[:n].values
    d["spread"] = d["best_ask"] - d["best_bid"]
    for c in ["market_buy_volume", "market_sell_volume", "limit_buy_volume",
              "limit_sell_volume", "withdraw_buy_volume", "withdraw_sell_volume",
              "market_buy_n", "market_sell_n", "limit_buy_n", "limit_sell_n",
              "withdraw_buy_n", "withdraw_sell_n"]:
        d[c] = msg[c].iloc[:n].values
    return d


def load_trades(cfg):
    tr = pd.read_csv(cfg["trades_path"])
    tr["timestamp"] = pd.to_datetime(tr["timestamp"])
    return tr.sort_values("timestamp").reset_index(drop=True)


def build_lob_array(data, nl=10):
    cols = []
    for i in range(1, nl + 1):
        cols += [f"ask{i}_price", f"ask{i}_volume",
                 f"bid{i}_price", f"bid{i}_volume"]
    return data[cols].values.astype(np.float32)


def fit_lob_normalizer(raw_lob, nl=10):
    x = raw_lob.copy().astype(np.float32)
    mid = (x[:, 0] + x[:, 2]) / 2 + 1e-8
    for i in range(nl):
        x[:, 4 * i] = x[:, 4 * i] / mid - 1.0
        x[:, 4 * i + 2] = x[:, 4 * i + 2] / mid - 1.0
    for i in range(nl):
        x[:, 4 * i + 1] = np.log1p(x[:, 4 * i + 1])
        x[:, 4 * i + 3] = np.log1p(x[:, 4 * i + 3])
    mean = x.mean(axis=0).astype(np.float32)
    std = (x.std(axis=0) + 1e-8).astype(np.float32)
    return mean, std


def apply_lob_normalizer(raw_lob, nl, mean, std):
    x = raw_lob.copy().astype(np.float32)
    mid = (x[:, 0] + x[:, 2]) / 2 + 1e-8
    for i in range(nl):
        x[:, 4 * i] = x[:, 4 * i] / mid - 1.0
        x[:, 4 * i + 2] = x[:, 4 * i + 2] / mid - 1.0
    for i in range(nl):
        x[:, 4 * i + 1] = np.log1p(x[:, 4 * i + 1])
        x[:, 4 * i + 3] = np.log1p(x[:, 4 * i + 3])
    return ((x - mean) / std).astype(np.float32)


def make_labels(midprice, k=100, alpha=1e-5):
    mp = pd.Series(midprice)
    price_past = mp.rolling(window=k).mean()
    price_future = mp.copy()
    price_future.iloc[:-k] = price_past.iloc[k:].values
    price_future.iloc[-k:] = np.nan
    pct = (price_future - price_past) / (price_past + 1e-12)
    y = np.full(len(mp), -1, dtype=np.int64)
    y[pct >= alpha] = 2    # up
    y[pct <= -alpha] = 0   # down
    y[(pct > -alpha) & (pct < alpha)] = 1  # stationary
    return y


# ═══════════════════════════════════════════════════════════════════════════
# 2. MODEL — SimpleLOB (replaces AttnLOB for small datasets)
# ═══════════════════════════════════════════════════════════════════════════
#
# AttnLOB: 241,704 params — designed for 5M+ samples (paper used 21 trading days)
# SimpleLOB: ~10,300 params — works with 5K-50K samples
#
# Architecture: 1D conv along time axis (treats each timestep as a 40-dim vector)
# → AdaptivePool → FC → 64-dim feature → 3-class head

class SimpleLOB(nn.Module):
    def __init__(self, T=50, n_feat=40):
        super().__init__()
        self.T = T
        self.feat_dim = 64

        self.conv = nn.Sequential(
            # (B, 40, T) → (B, 32, T)
            nn.Conv1d(n_feat, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            # (B, 32, T) → (B, 16, T)
            nn.Conv1d(32, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.01),

            # (B, 16, T) → (B, 16, 1)
            nn.AdaptiveAvgPool1d(1),
        )

        self.fc = nn.Sequential(
            nn.Linear(16, 64),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),
        )

        self.cls = nn.Linear(64, 3)

    def features(self, x):
        # x: (B, T, 40)
        h = x.permute(0, 2, 1)           # (B, 40, T)
        h = self.conv(h).squeeze(-1)      # (B, 16)
        return self.fc(h)                 # (B, 64)

    def forward(self, x):
        return self.cls(self.features(x))


# ═══════════════════════════════════════════════════════════════════════════
# 3. PRETRAIN
# ═══════════════════════════════════════════════════════════════════════════

class LOBDataset(Dataset):
    def __init__(self, raw_lob, labels, indices, T, mean, std, nl=10):
        self.raw = raw_lob
        self.labels = labels
        self.T = T
        self.mean, self.std, self.nl = mean, std, nl
        self.idx = indices

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        t = self.idx[i]
        window = self.raw[t - self.T:t]
        normed = apply_lob_normalizer(window, self.nl, self.mean, self.std)
        return (torch.tensor(normed, dtype=torch.float32),
                torch.tensor(self.labels[t], dtype=torch.long))


def pretrain(data, raw_lob, lob_mean, lob_std, cfg):
    print("── Pretrain SimpleLOB ──")
    labels = make_labels(data["midprice"].values, cfg["k"], cfg["alpha"])
    T, k, nl = cfg["T"], cfg["k"], cfg["n_levels"]

    # All valid indices
    all_valid = np.where(
        (np.arange(len(labels)) >= T) &
        (np.arange(len(labels)) < len(labels) - k) &
        (labels >= 0)
    )[0]

    n = len(all_valid)
    n_tr = int(0.7 * n)
    n_va = int(0.15 * n)
    gap = T + k   # minimum, can be larger

    tr_idx = all_valid[:n_tr]
    va_start = n_tr + gap
    va_end = min(va_start + n_va, n)
    va_idx = all_valid[va_start:va_end]
    te_start = va_end + gap
    te_idx = all_valid[te_start:]

    y_tr, y_va = labels[tr_idx], labels[va_idx]
    for name, ys in [("Train", y_tr), ("Val", y_va)]:
        cnt = np.bincount(ys, minlength=3)
        pct = 100 * cnt / cnt.sum()
        print(f"  {name}: n={len(ys)}, down={pct[0]:.1f}% stat={pct[1]:.1f}% up={pct[2]:.1f}%")

    tr_dl = DataLoader(
        LOBDataset(raw_lob, labels, tr_idx, T, lob_mean, lob_std, nl),
        cfg["batch_size"], shuffle=True, drop_last=True)
    va_dl = DataLoader(
        LOBDataset(raw_lob, labels, va_idx, T, lob_mean, lob_std, nl),
        cfg["batch_size"])

    model = SimpleLOB(T).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {n_params:,} (samples/params = {len(tr_idx)/n_params:.1f})")

    opt = optim.Adam(model.parameters(), cfg["pretrain_lr"], weight_decay=1e-4)

    # Inverse-frequency weights
    cnt = np.bincount(y_tr, minlength=3).astype(float) + 1
    w = 1.0 / cnt
    w = w / w.sum() * 3.0
    w = torch.tensor(w, dtype=torch.float32).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["pretrain_epochs"])

    best_f1, best_sd = 0, None
    patience, patience_limit = 0, 15

    for ep in range(cfg["pretrain_epochs"]):
        model.train()
        ep_loss, nb = 0, 0
        for xb, yb in tr_dl:
            logits = model(xb.to(DEVICE))
            loss = crit(logits, yb.to(DEVICE))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
            nb += 1
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in va_dl:
                preds.append(model(xb.to(DEVICE)).argmax(1).cpu())
                trues.append(yb)
        p = torch.cat(preds).numpy()
        t = torch.cat(trues).numpy()

        f1s = []
        for c in range(3):
            tp = ((p == c) & (t == c)).sum()
            fp = ((p == c) & (t != c)).sum()
            fn = ((p != c) & (t == c)).sum()
            pr = tp / (tp + fp + 1e-8)
            rc = tp / (tp + fn + 1e-8)
            f1s.append(2 * pr * rc / (pr + rc + 1e-8))
        f1 = np.mean(f1s)

        if f1 > best_f1:
            best_f1 = f1
            best_sd = {k_: v.cpu().clone() for k_, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        pd_dist = np.bincount(p, minlength=3)
        if ep % 5 == 0 or ep == cfg["pretrain_epochs"] - 1 or patience == 0:
            print(f"  ep {ep:3d}  loss={ep_loss / nb:.4f}  "
                  f"acc={np.mean(p == t):.3f}  F1={f1:.3f}  "
                  f"preds=[{pd_dist[0]},{pd_dist[1]},{pd_dist[2]}]  "
                  f"f1/cls=[{f1s[0]:.3f},{f1s[1]:.3f},{f1s[2]:.3f}]"
                  f"{'  *best*' if patience == 0 else ''}")

        if patience >= patience_limit:
            print(f"  Early stop at ep {ep}")
            break

    model.load_state_dict(best_sd)
    print(f"  Best F1 = {best_f1:.3f}")
    return model


# ═══════════════════════════════════════════════════════════════════════════
# 4. DYNAMIC FEATURES
# ═══════════════════════════════════════════════════════════════════════════

DYN_DIM = 24


def dynamic_features(data, idx, ts, cfg=None):
    if cfg is None:
        cfg = CFG
    f = []
    rv_rsi_win = cfg.get("rv_rsi_windows", [5, 15, 60])
    osi_win = cfg.get("osi_windows", [2, 10, 30])

    mp = data["midprice"].values[:idx + 1]

    for wsec in rv_rsi_win:
        mask = ts[:idx + 1] >= ts[idx] - wsec
        w = mp[max(0, idx + 1 - mask.sum()):]
        if len(w) > 1:
            lr = np.diff(np.log(w + 1e-12))
            f.append(np.sqrt((lr ** 2).sum()) * 1e4)
        else:
            f.append(0.0)

    for wsec in rv_rsi_win:
        mask = ts[:idx + 1] >= ts[idx] - wsec
        w = mp[max(0, idx + 1 - mask.sum()):]
        if len(w) > 1:
            d = np.diff(w)
            g, l = np.maximum(d, 0).sum(), np.maximum(-d, 0).sum()
            f.append(g / (g + l + 1e-8))
        else:
            f.append(0.5)

    for wsec in osi_win:
        mask = ts[:idx + 1] >= ts[idx] - wsec
        w = data.iloc[max(0, idx + 1 - mask.sum()):idx + 1]
        if len(w):
            mbv = w["market_buy_volume"].sum()
            msv = w["market_sell_volume"].sum()
            mbn = w["market_buy_n"].sum()
            msn = w["market_sell_n"].sum()
            lbv = w["limit_buy_volume"].sum()
            lsv = w["limit_sell_volume"].sum()
            lbn = w["limit_buy_n"].sum()
            lsn = w["limit_sell_n"].sum()
            cbv = w["withdraw_buy_volume"].sum()
            csv_ = w["withdraw_sell_volume"].sum()
            cbn = w["withdraw_buy_n"].sum()
            csn = w["withdraw_sell_n"].sum()
            f += [
                (mbv - msv) / (mbv + msv + 1e-7),
                (mbn - msn) / (mbn + msn + 1e-7),
                (lbv - lsv) / (lbv + lsv + 1e-7),
                (lbn - lsn) / (lbn + lsn + 1e-7),
                (cbv - csv_) / (cbv + csv_ + 1e-7),
                (cbn - csn) / (cbn + csn + 1e-7),
            ]
        else:
            f += [0] * 6

    return np.array(f, dtype=np.float32)


# ═══════════════════════════════════════════════════════════════════════════
# 5. ENVIRONMENT
# ═══════════════════════════════════════════════════════════════════════════

def price_legal_check(ask, bid, tick=0.01):
    ask = math.ceil(ask / tick) * tick
    bid = math.floor(bid / tick) * tick
    return round(ask, 2), round(bid, 2)


class MMEnv:
    def __init__(self, data, raw_lob, trades, cfg, lob_mean, lob_std):
        self.data = data
        self.raw_lob = raw_lob
        self.cfg = cfg
        self.lob_mean, self.lob_std = lob_mean, lob_std
        self.ts = (data["timestamp"] - data["timestamp"].iloc[0]).dt.total_seconds().values

        self.is_trade = np.zeros(len(data), dtype=bool)
        self.trades_by_ts = {}
        if trades is not None and len(trades):
            for ts_val, grp in trades.groupby("timestamp"):
                self.trades_by_ts[ts_val] = grp
            trade_ts_set = set(self.trades_by_ts.keys())
            for i, t in enumerate(data["timestamp"].values):
                if t in trade_ts_set:
                    self.is_trade[i] = True
        self.trades_df = trades

    def reset(self, start=None):
        c = self.cfg
        T = c["T"]
        mx = len(self.data) - c["episode_len"] - T - 1
        self.ep_start = start if start is not None else np.random.randint(T, max(T + 1, mx))
        self.ep_end = min(self.ep_start + c["episode_len"], len(self.data))

        ep_is_trade = self.is_trade[self.ep_start:self.ep_end]
        trade_idx = np.where(ep_is_trade)[0]
        trade_idx = trade_idx[trade_idx > T]
        self.trade_indices = iter(trade_idx)

        self.inv = 0
        self.cash = 0.0
        self.value = 0.0
        self.prev_value = 0.0
        self.n_trades = 0
        self.vol = 0
        self.inv_hist = []
        self.mid_price_prev = None

        try:
            self.i = next(self.trade_indices)
            self.i_next = next(self.trade_indices)
        except StopIteration:
            self.i = T
            self.i_next = None
        return self._obs()

    def _abs_idx(self):
        return self.ep_start + self.i

    def _get_price(self, rel_idx):
        row = self.data.iloc[self.ep_start + rel_idx]
        ba, bb = row["best_ask"], row["best_bid"]
        return (ba + bb) / 2, ba, bb, ba - bb

    def _obs(self):
        ai = self._abs_idx()
        c = self.cfg
        T, lat = c["T"], c["latency"]
        obs_i = max(0, ai - lat)
        raw = self.raw_lob[max(0, obs_i - T):obs_i]
        if len(raw) < T:
            raw = np.vstack([np.zeros((T - len(raw), raw.shape[1]), np.float32), raw])
        lob = apply_lob_normalizer(raw, c["n_levels"], self.lob_mean, self.lob_std)
        dyn = dynamic_features(self.data, obs_i, self.ts, c)
        inv_frac = self.inv / (c["max_inv"] * c["trade_unit"])
        time_frac = self.i / c["episode_len"]
        ag = np.array([inv_frac] * 12 + [time_frac] * 12, dtype=np.float32)
        self._last_obs = (lob, dyn, ag)
        return lob, dyn, ag

    def step(self, a1, a2):
        c = self.cfg
        tu, tick, lat = c["trade_unit"], c["tick"], c["latency"]
        mid, ba, bb, spread = self._get_price(self.i)
        if self.mid_price_prev is None:
            self.mid_price_prev = mid

        mid_lat, _, _, _ = self._get_price(self.i - lat)

        # Compute reservation price: skew toward reducing inventory
        delta = a1 * c["max_bias"]
        if self.inv > 0:
            pr = mid_lat - delta
        elif self.inv < 0:
            pr = mid_lat + delta
        else:
            pr = mid_lat

        sp = a2 * c["max_spread"]
        ask_p = pr + sp / 2
        bid_p = pr - sp / 2
        ask_p, bid_p = price_legal_check(ask_p, bid_p, tick)

        # Position limits
        if self.inv <= -c["max_inv"] * tu:
            ask_p = 0
        elif self.inv >= c["max_inv"] * tu:
            bid_p = 0

        trade_price, trade_volume = self._match(ask_p, bid_p, tu)
        self.inv += trade_volume
        self.cash -= trade_volume * trade_price
        self.value = self.cash + self.inv * mid
        pnl = self.value - self.prev_value

        dp = pnl - max(0.0, c["eta"] * pnl)
        tp = trade_volume * (mid - trade_price) if trade_volume else 0.0
        inv_lots = self.inv / c["trade_unit"]
        ip = c["zeta"] * (inv_lots ** 2)
        reward = dp + tp - ip

        self.prev_value = self.value
        self.mid_price_prev = mid
        if trade_volume != 0:
            self.n_trades += 1
            self.vol += abs(trade_volume * trade_price)
        self.inv_hist.append(self.inv)

        prev_i = self.i
        terminal = False
        self.i = self.i_next
        try:
            self.i_next = next(self.trade_indices)
        except StopIteration:
            terminal = True

        # Close position at episode end
        if terminal and self.inv:
            _, a1_close, b1_close, _ = self._get_price(prev_i)
            if self.inv > 0:
                tv, tp_close = -self.inv, b1_close
            else:
                tv, tp_close = -self.inv, a1_close
            self.inv += tv
            self.cash -= tv * tp_close
            self.value = self.cash + self.inv * mid
            reward += self.value - self.prev_value
            self.prev_value = self.value

        obs = self._obs() if not terminal else self._last_obs
        return obs, reward, terminal, {
            "val": self.value, "trades": self.n_trades, "inv": self.inv}

    def _match(self, ask_p, bid_p, tu):
        ai = self._abs_idx()
        ts = self.data.iloc[ai]["timestamp"]
        now_trades = self.trades_by_ts.get(ts)
        if now_trades is None or len(now_trades) == 0:
            return 0, 0

        buy_trades = now_trades[now_trades["aggressor_side"] == "B"]
        sell_trades = now_trades[now_trades["aggressor_side"] == "A"]
        _, a1_prev, b1_prev, _ = self._get_price(self.i - 1)

        trade_price, trade_volume = 0, 0

        # Our ASK fills against buy-aggressive trades
        if ask_p and ask_p > 0 and len(buy_trades):
            max_tp = buy_trades["price"].max()
            max_tv = buy_trades[buy_trades["price"] == max_tp]["size"].sum()
            if ask_p <= b1_prev:
                trade_price, trade_volume = b1_prev, -tu
            elif max_tp > ask_p:
                trade_price, trade_volume = ask_p, -tu
            elif max_tp == ask_p:
                depth = self.data.iloc[ai].get("ask1_volume", 100)
                prob = max_tv / (max_tv + depth + 1e-7)
                if np.random.random() < prob:
                    trade_price, trade_volume = ask_p, -tu

        # Our BID fills against sell-aggressive trades
        if bid_p and bid_p > 0 and len(sell_trades):
            min_tp = sell_trades["price"].min()
            min_tv = sell_trades[sell_trades["price"] == min_tp]["size"].sum()
            if bid_p >= a1_prev:
                trade_price, trade_volume = a1_prev, tu
            elif min_tp < bid_p:
                trade_price, trade_volume = bid_p, tu
            elif min_tp == bid_p:
                depth = self.data.iloc[ai].get("bid1_volume", 100)
                prob = min_tv / (min_tv + depth + 1e-7)
                if np.random.random() < prob:
                    trade_price, trade_volume = bid_p, tu

        return trade_price, trade_volume


# ═══════════════════════════════════════════════════════════════════════════
# 6. PPO POLICY
# ═══════════════════════════════════════════════════════════════════════════

class Policy(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.bb = backbone
        for p in self.bb.parameters():
            p.requires_grad = False

        feat = backbone.feat_dim   # 64
        self.trunk = nn.Sequential(
            nn.Linear(feat + DYN_DIM + 24, 64),
            nn.LeakyReLU(0.01),
        )
        self.alpha_head = nn.Linear(64, 2)
        self.beta_head = nn.Linear(64, 2)
        self.val = nn.Linear(64, 1)

    def forward(self, lob, dyn, ag):
        with torch.no_grad():
            f = self.bb.features(lob)
        h = self.trunk(torch.cat([f, dyn, ag], -1))
        alpha = F.softplus(self.alpha_head(h)) + 1.0
        beta = F.softplus(self.beta_head(h)) + 1.0
        value = self.val(h).squeeze(-1)
        return alpha, beta, value

    def act(self, lob, dyn, ag, det=False):
        alpha, beta, v = self(lob, dyn, ag)
        d = torch.distributions.Beta(alpha, beta)
        if det:
            a = alpha / (alpha + beta)
            return a, torch.zeros(a.shape[0], device=a.device), v
        a = d.rsample()
        logp = d.log_prob(a).sum(-1)
        return a, logp, v


# ═══════════════════════════════════════════════════════════════════════════
# 7. PPO TRAINING
# ═══════════════════════════════════════════════════════════════════════════

def gae(rews, vals, dones, gamma, lam):
    T = len(rews)
    adv = np.zeros(T, np.float32)
    g = nv = 0.0
    for t in reversed(range(T)):
        if dones[t]:
            nv = g = 0.0
        g = rews[t] + gamma * nv - vals[t] + gamma * lam * g
        adv[t] = g
        nv = vals[t]
    return adv, adv + np.array(vals)


def to_t(x):
    return torch.tensor(x, dtype=torch.float32, device=DEVICE)


def train(env, policy, cfg):
    print(f"\n── PPO ({cfg['rl_epochs']} ep × {cfg['eps_per_epoch']} traj) ──")
    opt = optim.Adam(
        filter(lambda p: p.requires_grad, policy.parameters()),
        cfg["rl_lr"])

    for epoch in range(cfg["rl_epochs"]):
        obs_l, obs_d, obs_a = [], [], []
        acts, logps, rews, vals, dns = [], [], [], [], []
        ep_r, ep_tr, ep_inv = [], [], []

        policy.eval()
        for _ in range(cfg["eps_per_epoch"]):
            lob, dyn, ag = env.reset()
            er = 0
            while True:
                lt = to_t(lob).unsqueeze(0)
                dt = to_t(dyn).unsqueeze(0)
                at = to_t(ag).unsqueeze(0)
                with torch.no_grad():
                    act, lp, v = policy.act(lt, dt, at)
                an = act.squeeze(0).cpu().numpy()
                (lob2, dyn2, ag2), r, done, info = env.step(an[0], an[1])
                obs_l.append(lob); obs_d.append(dyn); obs_a.append(ag)
                acts.append(an); logps.append(lp.item())
                rews.append(r); vals.append(v.item()); dns.append(done)
                er += r
                lob, dyn, ag = lob2, dyn2, ag2
                if done:
                    break
            ep_r.append(er)
            ep_tr.append(info["trades"])
            if env.inv_hist:
                ep_inv.append(np.mean(np.abs(env.inv_hist)))

        if not rews:
            continue

        adv, ret = gae(rews, vals, dns, cfg["gamma"], cfg["lam"])
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        policy.train()
        N = len(rews)
        idx = np.arange(N)
        Tl = to_t(np.stack(obs_l))
        Td = to_t(np.stack(obs_d))
        Ta = to_t(np.stack(obs_a))
        Tact = to_t(np.stack(acts))
        Tolp = to_t(logps)
        Tadv = to_t(adv)
        Tret = to_t(ret)

        for _ in range(cfg["ppo_updates"]):
            np.random.shuffle(idx)
            for s in range(0, N, cfg["mb_size"]):
                mb = idx[s:s + cfg["mb_size"]]
                alpha, beta, v = policy(Tl[mb], Td[mb], Ta[mb])
                dist = torch.distributions.Beta(alpha, beta)
                acts_mb = Tact[mb].clamp(1e-6, 1 - 1e-6)
                nlp = dist.log_prob(acts_mb).sum(-1)
                ratio = (nlp - Tolp[mb]).exp()
                cl = ratio.clamp(1 - cfg["ppo_clip"], 1 + cfg["ppo_clip"])
                pg_loss = -torch.min(ratio * Tadv[mb], cl * Tadv[mb]).mean()
                v_loss = 0.5 * F.mse_loss(v, Tret[mb])
                entropy = dist.entropy().mean()
                loss = pg_loss + v_loss - 0.01 * entropy
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
                opt.step()

        if epoch % 5 == 0 or epoch == cfg["rl_epochs"] - 1:
            avg_inv = np.mean(ep_inv) if ep_inv else 0
            print(f"  ep {epoch:4d}  rew={np.mean(ep_r):+8.2f}  "
                  f"trades={np.mean(ep_tr):5.1f}  "
                  f"avg|inv|={avg_inv:.1f}")

    return policy
    # Run
if __name__ == "__main__":
    data = load_data(CFG)
    trades = load_trades(CFG)
    raw_lob = build_lob_array(data, CFG["n_levels"])

    # fit normalizer on training portion only
    n_tr = int(len(raw_lob) * 0.8)
    lob_mean, lob_std = fit_lob_normalizer(raw_lob[:n_tr], CFG["n_levels"])
    print(f"LOB: {len(data)} events, Trades: {len(trades)} fills, device={DEVICE}")

    backbone = pretrain(data, raw_lob, lob_mean, lob_std, CFG)
    env = MMEnv(data, raw_lob, trades, CFG, lob_mean, lob_std)
    policy = Policy(backbone).to(DEVICE)
    print(f"Trainable: {sum(p.numel() for p in policy.parameters() if p.requires_grad):,} params")

    policy = train(env, policy, CFG)
    torch.save(policy.state_dict(), "mm_policy.pt")
    print("Saved mm_policy.pt")


LOB: 49736 events, Trades: 1061 fills, device=cuda
── Pretrain SimpleLOB ──
  Train: n=34675, down=38.2% stat=25.8% up=36.0%
  Val: n=7430, down=28.7% stat=26.5% up=44.8%
  Model params: 10,387 (samples/params = 3.3)
  ep   0  loss=0.8451  acc=0.399  F1=0.334  preds=[3967,761,2702]  f1/cls=[0.431,0.043,0.529]  *best*
  ep   3  loss=0.4489  acc=0.415  F1=0.405  preds=[3725,1465,2240]  f1/cls=[0.403,0.334,0.478]  *best*
  ep   5  loss=0.3985  acc=0.366  F1=0.325  preds=[3831,1022,2577]  f1/cls=[0.358,0.119,0.499]
  ep   6  loss=0.3788  acc=0.433  F1=0.418  preds=[3648,1416,2366]  f1/cls=[0.416,0.324,0.514]  *best*
  ep  10  loss=0.3180  acc=0.422  F1=0.416  preds=[3703,1509,2218]  f1/cls=[0.411,0.366,0.469]
  ep  12  loss=0.3012  acc=0.427  F1=0.423  preds=[3613,1850,1967]  f1/cls=[0.422,0.390,0.457]  *best*
  ep  14  loss=0.2864  acc=0.443  F1=0.437  preds=[3622,1736,2072]  f1/cls=[0.438,0.386,0.488]  *best*
  ep  15  loss=0.2808  acc=0.425  F1=0.419  preds=[3807,1609,2014]  f1/cls=[0.4

KeyboardInterrupt: 

In [ ]:
"""
Market-Making RL Agent — v3 (Complete Rewrite)
================================================
Previous attempts failed due to TWO independent bugs:

PRETRAIN BUG: AttnLOB has 241K params but only ~4K effective training samples.
  Deep CNN-attention gets stuck outputting a single class (loss flat at 1.099).
  FIX: SimpleLOB with 10K params — 24× smaller. Uses 1D convolutions that can
  actually learn from limited data.

RL BUG: max_spread=0.20 means agent quotes ±$0.05 from mid.
  But AAPL buy trades only reach mid+$0.015 at the 99th percentile.
  Agent NEVER gets filled → 0 trades per episode.
  FIX: max_spread=0.03 (quotes at ±$0.0075 initially → fills possible).
"""

import numpy as np
import pandas as pd
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

CFG = dict(
    ask_path="/content/ask.csv",       # ask1..10 price/vol
    bid_path="/content/bid.csv",       # bid1..10 price/vol
    msg_path="/content/msg.csv",       # message flow
    bbo_path="/content/price.csv",     # ask1_price, bid1_price, midprice
    trades_path="/content/trades.csv", # price, size, aggressor_side

    n_levels=10,
    T=50,
    tick=0.01,
    k=100,                # ~600ms horizon at 6ms/event (matches paper's ~1s)
    alpha=1e-5,

    # Pretrain
    pretrain_epochs=60,
    pretrain_lr=1e-3,
    batch_size=128,

    # RL env
    episode_len=2000,
    max_inv=10,
    trade_unit=10,

    # CRITICAL: calibrated for AAPL $265 with $0.02 spread
    # Buy trades reach mid+$0.015 at 99th pctl, mid+$0.01 at 90th pctl
    max_bias=0.01,         # max skew = 1 tick ($0.01)
    max_spread=0.03,       # initial quotes at ±$0.0075 → fills at ~70th pctl

    eta=0.5,
    zeta=1e-4,
    latency=1,

    # RL training
    rl_epochs=150,
    eps_per_epoch=32,
    rl_lr=3e-4,
    gamma=0.99,
    lam=0.95,
    ppo_clip=0.2,
    ppo_updates=4,
    mb_size=256,

    # Dynamic feature windows for 5-min dataset
    rv_rsi_windows=[5, 15, 60],
    osi_windows=[2, 10, 30],
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data

def load_data(cfg):
    ask = pd.read_csv(cfg["ask_path"])
    bid = pd.read_csv(cfg["bid_path"])
    msg = pd.read_csv(cfg["msg_path"])
    bbo = pd.read_csv(cfg["bbo_path"])
    n = min(len(ask), len(bid), len(msg), len(bbo))
    d = pd.DataFrame({"timestamp": pd.to_datetime(ask["timestamp"].iloc[:n])})
    for i in range(1, cfg["n_levels"] + 1):
        d[f"ask{i}_price"] = ask[f"ask{i}_price"].iloc[:n].values
        d[f"ask{i}_volume"] = ask[f"ask{i}_volume"].iloc[:n].values
        d[f"bid{i}_price"] = bid[f"bid{i}_price"].iloc[:n].values
        d[f"bid{i}_volume"] = bid[f"bid{i}_volume"].iloc[:n].values
    d["midprice"] = bbo["midprice"].iloc[:n].values
    d["best_ask"] = bbo["ask1_price"].iloc[:n].values
    d["best_bid"] = bbo["bid1_price"].iloc[:n].values
    d["spread"] = d["best_ask"] - d["best_bid"]
    for c in ["market_buy_volume", "market_sell_volume", "limit_buy_volume",
              "limit_sell_volume", "withdraw_buy_volume", "withdraw_sell_volume",
              "market_buy_n", "market_sell_n", "limit_buy_n", "limit_sell_n",
              "withdraw_buy_n", "withdraw_sell_n"]:
        d[c] = msg[c].iloc[:n].values
    return d


def load_trades(cfg):
    tr = pd.read_csv(cfg["trades_path"])
    tr["timestamp"] = pd.to_datetime(tr["timestamp"])
    return tr.sort_values("timestamp").reset_index(drop=True)


def build_lob_array(data, nl=10):
    cols = []
    for i in range(1, nl + 1):
        cols += [f"ask{i}_price", f"ask{i}_volume",
                 f"bid{i}_price", f"bid{i}_volume"]
    return data[cols].values.astype(np.float32)

# Normalization
# Do this such that prices become stationary looking, because they are expressed as percentages around the current mid instead of raw absolute prices.
# Not 1-to-1 as paper, but the volumes are normalized using log1p + z-norm (rather than volume/max)

def _normalize_lob_raw(x, nl):
    mid = (x[:, 0] + x[:, 2]) / 2 + 1e-8
    for i in range(nl):
        x[:, 4 * i] = x[:, 4 * i] / mid - 1.0
        x[:, 4 * i + 2] = x[:, 4 * i + 2] / mid - 1.0
        x[:, 4 * i + 1] = np.log1p(x[:, 4 * i + 1])
        x[:, 4 * i + 3] = np.log1p(x[:, 4 * i + 3])
    return x

def fit_lob_stats(raw_lob, nl=10):
    x = _normalize_lob_raw(raw_lob.copy().astype(np.float32), nl)
    return x.mean(axis=0), x.std(axis=0) + 1e-8


def apply_lob_normalizer(raw_lob, nl, mean, std):
    x = _normalize_lob_raw(raw_lob.copy().astype(np.float32), nl)
    return (x - mean) / std


# Labels
# Creates the target for pretraining -> turn the raw midprice series into a supervised learning problem:
# Given the last 50 LOB states, predict whether price will go down / stay / go up.

def make_labels(midprice, k=100, alpha=1e-5):
    mp = pd.Series(midprice)
    past_avg = mp.rolling(k).mean()
    future_avg = past_avg.shift(-k)
    pct = (future_avg - past_avg) / (past_avg + 1e-12)

    y = np.full(len(mp), -1, dtype=np.int64)
    y[pct >= alpha] = 2    # up
    y[pct <= -alpha] = 0   # down
    y[(pct > -alpha) & (pct < alpha)] = 1  # stationary
    return y


# ═══════════════════════════════════════════════════════════════════════════
# MODEL — SimpleLOB (replaces AttnLOB for small datasets)
# ═══════════════════════════════════════════════════════════════════════════
#
# AttnLOB: 241,704 params — designed for 5M+ samples (paper used 21 trading days)
# SimpleLOB: ~10,300 params — works with 5K-50K samples
#
# Architecture: 1D conv along time axis (treats each timestep as a 40-dim vector)
# → AdaptivePool → FC → 64-dim feature → 3-class head

class SimpleLOB(nn.Module):
    def __init__(self, T=50, n_feat=40):
        super().__init__()
        self.T = T
        self.feat_dim = 64

        self.conv = nn.Sequential(
            # (B, 40, T) → (B, 32, T)
            nn.Conv1d(n_feat, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            # (B, 32, T) → (B, 16, T)
            nn.Conv1d(32, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.01),

            # (B, 16, T) → (B, 16, 1)
            nn.AdaptiveAvgPool1d(1),
        )

        self.fc = nn.Sequential(
            nn.Linear(16, 64),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),
        )

        self.cls = nn.Linear(64, 3)

    def features(self, x):
        # x: (B, T, 40) where B is batch size, T is lookback length and 40 raw LOB features
        h = x.permute(0, 2, 1)           # (B, 40, T)
        h = self.conv(h).squeeze(-1)      # (B, 16)
        return self.fc(h)                 # (B, 64)

    def forward(self, x):
        return self.cls(self.features(x))


# Pretrain
class LOBDataset(Dataset):
    def __init__(self, raw_lob, labels, indices, T, mean, std, nl=10):
        self.raw = raw_lob
        self.labels = labels
        self.indices = np.asarray(indices, dtype=np.int64)
        self.T = T
        self.mean = mean
        self.std = std
        self.nl = nl

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        window = self.raw[t - self.T:t]
        normed = apply_lob_normalizer(window, self.nl, self.mean, self.std)
        return (
            torch.tensor(normed, dtype=torch.float32),
            torch.tensor(self.labels[t], dtype=torch.long),
        )

def make_pretrain_splits(labels, T, k, train_frac=0.7, val_frac=0.15, gap=None):
    """
    Chronological split with an embargo gap to reduce leakage from overlapping
    LOB windows and rolling labels.

    Returns:
        tr_idx, va_idx, te_idx, gap
    """
    if gap is None:
        gap = T + k

    all_valid = np.where(
        (np.arange(len(labels)) >= T) &
        (np.arange(len(labels)) < len(labels) - k) &
        (labels >= 0)
    )[0]

    n = len(all_valid)
    if n == 0:
        raise ValueError("No valid indices for pretraining.")

    n_tr = int(train_frac * n)
    n_va = int(val_frac * n)

    tr_idx = all_valid[:n_tr]

    va_start = n_tr + gap
    va_end = min(va_start + n_va, n)
    va_idx = all_valid[va_start:va_end]

    te_start = va_end + gap
    te_idx = all_valid[te_start:]

    if len(tr_idx) == 0:
        raise ValueError("Training split is empty.")
    if len(va_idx) == 0:
        raise ValueError("Validation split is empty. Reduce gap or change split fractions.")

    return tr_idx, va_idx, te_idx, gap

def pretrain(data, raw_lob, cfg):
    print("── Pretrain SimpleLOB ──")

    labels = make_labels(data["midprice"].values, cfg["k"], cfg["alpha"])
    T = cfg["T"]
    k = cfg["k"]
    nl = cfg["n_levels"]

    # Chronological split with embargo gap
    tr_idx, va_idx, te_idx, gap = make_pretrain_splits(
        labels,
        T=T,
        k=k,
        train_frac=0.7,
        val_frac=0.15,
        gap=T + k,
    )

    print(f"  Valid indices: {len(tr_idx) + len(va_idx) + len(te_idx)}")
    print(f"  Gap: {gap}")
    print(f"  Train: {len(tr_idx)}  Val: {len(va_idx)}  Test: {len(te_idx)}")

    # Fit normalization ONLY on the training region
    # Since tr_idx is chronological, this safely covers the train period.
    train_end_row = tr_idx[-1] + 1
    lob_mean, lob_std = fit_lob_normalizer(raw_lob[:train_end_row], nl)

    y_tr = labels[tr_idx]
    y_va = labels[va_idx]

    for name, ys in [("Train", y_tr), ("Val", y_va)]:
        cnt = np.bincount(ys, minlength=3)
        pct = 100.0 * cnt / cnt.sum()
        print(
            f"  {name}: n={len(ys)}, "
            f"down={pct[0]:.1f}% stat={pct[1]:.1f}% up={pct[2]:.1f}%"
        )

    tr_dl = DataLoader(
        LOBDataset(raw_lob, labels, tr_idx, T, lob_mean, lob_std, nl),
        batch_size=cfg["batch_size"],
        shuffle=True,
        drop_last=True,
    )
    va_dl = DataLoader(
        LOBDataset(raw_lob, labels, va_idx, T, lob_mean, lob_std, nl),
        batch_size=cfg["batch_size"],
        shuffle=False,
    )

    model = SimpleLOB(T).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {n_params:,} (samples/params = {len(tr_idx) / n_params:.1f})")

    opt = optim.Adam(model.parameters(), cfg["pretrain_lr"], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["pretrain_epochs"])

    # Class weights from TRAIN ONLY
    cnt = np.bincount(y_tr, minlength=3).astype(np.float32) + 1.0
    w = 1.0 / cnt
    w = w / w.sum() * 3.0
    w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)

    best_f1 = -1.0
    best_sd = None
    patience = 0
    patience_limit = 15

    for ep in range(cfg["pretrain_epochs"]):
        model.train()
        ep_loss = 0.0
        nb = 0

        for xb, yb in tr_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = crit(logits, yb)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            ep_loss += loss.item()
            nb += 1

        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in va_dl:
                xb = xb.to(DEVICE)
                logits = model(xb)
                preds.append(logits.argmax(1).cpu())
                trues.append(yb)

        p = torch.cat(preds).numpy()
        t = torch.cat(trues).numpy()

        f1s = []
        for c in range(3):
            tp = ((p == c) & (t == c)).sum()
            fp = ((p == c) & (t != c)).sum()
            fn = ((p != c) & (t == c)).sum()
            pr = tp / (tp + fp + 1e-8)
            rc = tp / (tp + fn + 1e-8)
            f1s.append(2 * pr * rc / (pr + rc + 1e-8))
        f1 = float(np.mean(f1s))
        acc = float((p == t).mean())

        if f1 > best_f1:
            best_f1 = f1
            best_sd = {k_: v.cpu().clone() for k_, v in model.state_dict().items()}
            patience = 0
            best_tag = "  *best*"
        else:
            patience += 1
            best_tag = ""

        pd_dist = np.bincount(p, minlength=3)

        if ep % 5 == 0 or ep == cfg["pretrain_epochs"] - 1 or patience == 0:
            print(
                f"  ep {ep:3d}  loss={ep_loss / max(nb, 1):.4f}  "
                f"acc={acc:.3f}  F1={f1:.3f}  "
                f"preds=[{pd_dist[0]},{pd_dist[1]},{pd_dist[2]}]  "
                f"f1/cls=[{f1s[0]:.3f},{f1s[1]:.3f},{f1s[2]:.3f}]"
                f"{best_tag}"
            )

        if patience >= patience_limit:
            print(f"  Early stop at ep {ep}")
            break

    if best_sd is None:
        raise RuntimeError("Pretraining failed: no best model state was saved.")

    model.load_state_dict(best_sd)
    print(f"  Best F1 = {best_f1:.3f}")

    return model, lob_mean, lob_std, (tr_idx, va_idx, te_idx)


# Dynamic features

DYN_DIM = 24 # 6 market state + 18 OSI

def dynamic_features(data, idx, ts_sec, cfg):
    mid = data["midprice"].values[:idx + 1]
    feats = []

    for w in cfg["rv_rsi_windows"]:
        start = max(0, idx + 1 - int((ts_sec[idx] - ts_sec[:idx+1] <= w).sum()))
        window = mid[start:]

        # Realized volatility
        if len(window) > 1:
            lr = np.diff(np.log(window + 1e-12))
            feats.append(np.sqrt((lr**2).sum()) * 1e4)
        else:
            feats.append(0.0)

        # RSI
        if len(window) > 1:
            d = np.diff(window)
            gains, losses = np.maximum(d, 0).sum(), np.maximum(-d, 0).sum()
            feats.append(gains / (gains + losses + 1e-8))
        else:
            feats.append(0.5)

    for w in cfg["osi_windows"]:
        start = max(0, idx + 1 - int((ts_sec[idx] - ts_sec[:idx+1] <= w).sum()))
        chunk = data.iloc[start:idx + 1]
        if len(chunk) == 0:
            feats.extend([0.0] * 6)
            continue
        # OSI: (buy - sell) / (buy + sell) for market/limit/cancel × vol/count
        for action in ("market", "limit", "withdraw"):
            for stat in ("volume", "n"):
                buy = chunk[f"{action}_buy_{stat}"].sum()
                sell = chunk[f"{action}_sell_{stat}"].sum()
                feats.append((buy - sell) / (buy + sell + 1e-7))

    return np.array(feats, dtype=np.float32)


# RL Environment (matching author's base_env.py + env_continuous.py)

def price_legal_check(ask, bid, tick=0.01):
    ask = math.ceil(ask / tick) * tick
    bid = math.floor(bid / tick) * tick
    return round(ask, 2), round(bid, 2)


class MMEnv:
    def __init__(self, data, raw_lob, trades, cfg, lob_mean, lob_std):
        self.data = data
        self.raw_lob = raw_lob
        self.cfg = cfg
        self.lob_mean, self.lob_std = lob_mean, lob_std
        self.ts = (data["timestamp"] - data["timestamp"].iloc[0]).dt.total_seconds().values

        self.is_trade = np.zeros(len(data), dtype=bool)
        self.trades_by_ts = {}
        if trades is not None and len(trades):
            for ts_val, grp in trades.groupby("timestamp"):
                self.trades_by_ts[ts_val] = grp
            trade_ts_set = set(self.trades_by_ts.keys())
            for i, t in enumerate(data["timestamp"].values):
                if t in trade_ts_set:
                    self.is_trade[i] = True
        self.trades_df = trades

    def reset(self, start=None):
        c = self.cfg
        T = c["T"]
        mx = len(self.data) - c["episode_len"] - T - 1
        self.ep_start = start if start is not None else np.random.randint(T, max(T + 1, mx))
        self.ep_end = min(self.ep_start + c["episode_len"], len(self.data))

        ep_is_trade = self.is_trade[self.ep_start:self.ep_end]
        trade_idx = np.where(ep_is_trade)[0]
        trade_idx = trade_idx[trade_idx > T]
        self.trade_indices = iter(trade_idx)

        self.inv = 0
        self.cash = 0.0
        self.value = 0.0
        self.prev_value = 0.0
        self.n_trades = 0
        self.vol = 0
        self.inv_hist = []
        self.mid_price_prev = None

        try:
            self.i = next(self.trade_indices)
            self.i_next = next(self.trade_indices)
        except StopIteration:
            self.i = T
            self.i_next = None
        return self._obs()

    def _abs_idx(self):
        return self.ep_start + self.i

    def _get_price(self, rel_idx):
        row = self.data.iloc[self.ep_start + rel_idx]
        ba, bb = row["best_ask"], row["best_bid"]
        return (ba + bb) / 2, ba, bb, ba - bb

    def _obs(self):
        ai = self._abs_idx()
        c = self.cfg
        T, lat = c["T"], c["latency"]
        obs_i = max(0, ai - lat)
        raw = self.raw_lob[max(0, obs_i - T):obs_i]
        if len(raw) < T:
            raw = np.vstack([np.zeros((T - len(raw), raw.shape[1]), np.float32), raw])
        lob = apply_lob_normalizer(raw, c["n_levels"], self.lob_mean, self.lob_std)
        dyn = dynamic_features(self.data, obs_i, self.ts, c)
        inv_frac = self.inv / (c["max_inv"] * c["trade_unit"])
        time_frac = self.i / c["episode_len"]
        ag = np.array([inv_frac] * 12 + [time_frac] * 12, dtype=np.float32)
        self._last_obs = (lob, dyn, ag)
        return lob, dyn, ag

    def step(self, a1, a2):
        c = self.cfg
        tu, tick, lat = c["trade_unit"], c["tick"], c["latency"]
        mid, ba, bb, spread = self._get_price(self.i)
        if self.mid_price_prev is None:
            self.mid_price_prev = mid

        mid_lat, _, _, _ = self._get_price(self.i - lat)

        # Compute reservation price: skew toward reducing inventory
        delta = a1 * c["max_bias"]
        if self.inv > 0:
            pr = mid_lat - delta
        elif self.inv < 0:
            pr = mid_lat + delta
        else:
            pr = mid_lat

        sp = a2 * c["max_spread"]
        ask_p = pr + sp / 2
        bid_p = pr - sp / 2
        ask_p, bid_p = price_legal_check(ask_p, bid_p, tick)

        # Position limits
        if self.inv <= -c["max_inv"] * tu:
            ask_p = 0
        elif self.inv >= c["max_inv"] * tu:
            bid_p = 0

        trade_price, trade_volume = self._match(ask_p, bid_p, tu)
        self.inv += trade_volume
        self.cash -= trade_volume * trade_price
        self.value = self.cash + self.inv * mid
        pnl = self.value - self.prev_value

        dp = pnl - max(0.0, c["eta"] * pnl)
        tp = trade_volume * (mid - trade_price) if trade_volume else 0.0
        inv_lots = self.inv / c["trade_unit"]
        ip = c["zeta"] * (inv_lots ** 2)
        reward = dp + tp - ip

        self.prev_value = self.value
        self.mid_price_prev = mid
        if trade_volume != 0:
            self.n_trades += 1
            self.vol += abs(trade_volume * trade_price)
        self.inv_hist.append(self.inv)

        prev_i = self.i
        terminal = False
        self.i = self.i_next
        try:
            self.i_next = next(self.trade_indices)
        except StopIteration:
            terminal = True

        # Close position at episode end
        if terminal and self.inv:
            _, a1_close, b1_close, _ = self._get_price(prev_i)
            if self.inv > 0:
                tv, tp_close = -self.inv, b1_close
            else:
                tv, tp_close = -self.inv, a1_close
            self.inv += tv
            self.cash -= tv * tp_close
            self.value = self.cash + self.inv * mid
            reward += self.value - self.prev_value
            self.prev_value = self.value

        obs = self._obs() if not terminal else self._last_obs
        return obs, reward, terminal, {
            "val": self.value, "trades": self.n_trades, "inv": self.inv}

    def _match(self, ask_p, bid_p, tu):
        ai = self._abs_idx()
        ts = self.data.iloc[ai]["timestamp"]
        now_trades = self.trades_by_ts.get(ts)
        if now_trades is None or len(now_trades) == 0:
            return 0, 0

        buy_trades = now_trades[now_trades["aggressor_side"] == "B"]
        sell_trades = now_trades[now_trades["aggressor_side"] == "A"]
        _, a1_prev, b1_prev, _ = self._get_price(self.i - 1)

        trade_price, trade_volume = 0, 0

        # Our ASK fills against buy-aggressive trades
        if ask_p and ask_p > 0 and len(buy_trades):
            max_tp = buy_trades["price"].max()
            max_tv = buy_trades[buy_trades["price"] == max_tp]["size"].sum()
            if ask_p <= b1_prev:
                trade_price, trade_volume = b1_prev, -tu
            elif max_tp > ask_p:
                trade_price, trade_volume = ask_p, -tu
            elif max_tp == ask_p:
                depth = self.data.iloc[ai].get("ask1_volume", 100)
                prob = max_tv / (max_tv + depth + 1e-7)
                if np.random.random() < prob:
                    trade_price, trade_volume = ask_p, -tu

        # Our BID fills against sell-aggressive trades
        if bid_p and bid_p > 0 and len(sell_trades):
            min_tp = sell_trades["price"].min()
            min_tv = sell_trades[sell_trades["price"] == min_tp]["size"].sum()
            if bid_p >= a1_prev:
                trade_price, trade_volume = a1_prev, tu
            elif min_tp < bid_p:
                trade_price, trade_volume = bid_p, tu
            elif min_tp == bid_p:
                depth = self.data.iloc[ai].get("bid1_volume", 100)
                prob = min_tv / (min_tv + depth + 1e-7)
                if np.random.random() < prob:
                    trade_price, trade_volume = bid_p, tu

        return trade_price, trade_volume


# PPO Policy

class Policy(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.bb = backbone
        for p in self.bb.parameters():
            p.requires_grad = False

        feat = backbone.feat_dim   # 64
        self.trunk = nn.Sequential(
            nn.Linear(feat + DYN_DIM + 24, 64),
            nn.LeakyReLU(0.01),
        )
        self.alpha_head = nn.Linear(64, 2)
        self.beta_head = nn.Linear(64, 2)
        self.val = nn.Linear(64, 1)

    def forward(self, lob, dyn, ag):
        with torch.no_grad():
            f = self.bb.features(lob)
        h = self.trunk(torch.cat([f, dyn, ag], -1))
        alpha = F.softplus(self.alpha_head(h)) + 1.0
        beta = F.softplus(self.beta_head(h)) + 1.0
        value = self.val(h).squeeze(-1)
        return alpha, beta, value

    def act(self, lob, dyn, ag, det=False):
        alpha, beta, v = self(lob, dyn, ag)
        d = torch.distributions.Beta(alpha, beta)
        if det:
            a = alpha / (alpha + beta)
            return a, torch.zeros(a.shape[0], device=a.device), v
        a = d.rsample()
        logp = d.log_prob(a).sum(-1)
        return a, logp, v

# PPO training

def gae(rews, vals, dones, gamma, lam):
    T = len(rews)
    adv = np.zeros(T, np.float32)
    g = nv = 0.0
    for t in reversed(range(T)):
        if dones[t]:
            nv = g = 0.0
        g = rews[t] + gamma * nv - vals[t] + gamma * lam * g
        adv[t] = g
        nv = vals[t]
    return adv, adv + np.array(vals)


def to_t(x):
    return torch.tensor(x, dtype=torch.float32, device=DEVICE)


def train(env, policy, cfg):
    print(f"\n── PPO ({cfg['rl_epochs']} ep × {cfg['eps_per_epoch']} traj) ──")
    opt = optim.Adam(
        filter(lambda p: p.requires_grad, policy.parameters()),
        cfg["rl_lr"])

    for epoch in range(cfg["rl_epochs"]):
        obs_l, obs_d, obs_a = [], [], []
        acts, logps, rews, vals, dns = [], [], [], [], []
        ep_r, ep_tr, ep_inv = [], [], []

        policy.eval()
        for _ in range(cfg["eps_per_epoch"]):
            lob, dyn, ag = env.reset()
            er = 0
            while True:
                lt = to_t(lob).unsqueeze(0)
                dt = to_t(dyn).unsqueeze(0)
                at = to_t(ag).unsqueeze(0)
                with torch.no_grad():
                    act, lp, v = policy.act(lt, dt, at)
                an = act.squeeze(0).cpu().numpy()
                (lob2, dyn2, ag2), r, done, info = env.step(an[0], an[1])
                obs_l.append(lob); obs_d.append(dyn); obs_a.append(ag)
                acts.append(an); logps.append(lp.item())
                rews.append(r); vals.append(v.item()); dns.append(done)
                er += r
                lob, dyn, ag = lob2, dyn2, ag2
                if done:
                    break
            ep_r.append(er)
            ep_tr.append(info["trades"])
            if env.inv_hist:
                ep_inv.append(np.mean(np.abs(env.inv_hist)))

        if not rews:
            continue

        adv, ret = gae(rews, vals, dns, cfg["gamma"], cfg["lam"])
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        policy.train()
        N = len(rews)
        idx = np.arange(N)
        Tl = to_t(np.stack(obs_l))
        Td = to_t(np.stack(obs_d))
        Ta = to_t(np.stack(obs_a))
        Tact = to_t(np.stack(acts))
        Tolp = to_t(logps)
        Tadv = to_t(adv)
        Tret = to_t(ret)

        for _ in range(cfg["ppo_updates"]):
            np.random.shuffle(idx)
            for s in range(0, N, cfg["mb_size"]):
                mb = idx[s:s + cfg["mb_size"]]
                alpha, beta, v = policy(Tl[mb], Td[mb], Ta[mb])
                dist = torch.distributions.Beta(alpha, beta)
                acts_mb = Tact[mb].clamp(1e-6, 1 - 1e-6)
                nlp = dist.log_prob(acts_mb).sum(-1)
                ratio = (nlp - Tolp[mb]).exp()
                cl = ratio.clamp(1 - cfg["ppo_clip"], 1 + cfg["ppo_clip"])
                pg_loss = -torch.min(ratio * Tadv[mb], cl * Tadv[mb]).mean()
                v_loss = 0.5 * F.mse_loss(v, Tret[mb])
                entropy = dist.entropy().mean()
                loss = pg_loss + v_loss - 0.01 * entropy
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
                opt.step()

        if epoch % 5 == 0 or epoch == cfg["rl_epochs"] - 1:
            avg_inv = np.mean(ep_inv) if ep_inv else 0
            print(f"  ep {epoch:4d}  rew={np.mean(ep_r):+8.2f}  "
                  f"trades={np.mean(ep_tr):5.1f}  "
                  f"avg|inv|={avg_inv:.1f}")

    return policy

# Run
if __name__ == "__main__":
    data = load_data(CFG)
    trades = load_trades(CFG)
    raw_lob = build_lob_array(data, CFG["n_levels"])

    print(f"LOB: {len(data)} events, Trades: {len(trades)} fills, device={DEVICE}")

    backbone, lob_mean, lob_std, splits = pretrain(data, raw_lob, CFG)

    env = MMEnv(data, raw_lob, trades, CFG, lob_mean, lob_std)
    policy = Policy(backbone).to(DEVICE)

    print(f"Trainable: {sum(p.numel() for p in policy.parameters() if p.requires_grad):,} params")

    policy = train(env, policy, CFG)
    torch.save(policy.state_dict(), "mm_policy.pt")
    print("Saved mm_policy.pt")


LOB: 49736 events, Trades: 1061 fills, device=cuda
── Pretrain SimpleLOB ──
  Valid indices: 49237
  Gap: 150
  Train: 34675  Val: 7430  Test: 7132
  Train: n=34675, down=38.2% stat=25.8% up=36.0%
  Val: n=7430, down=28.7% stat=26.5% up=44.8%
  Model params: 10,387 (samples/params = 3.3)
  ep   0  loss=0.8415  acc=0.379  F1=0.322  preds=[4740,755,1935]  f1/cls=[0.422,0.052,0.492]  *best*
  ep   1  loss=0.6065  acc=0.418  F1=0.400  preds=[3894,1370,2166]  f1/cls=[0.430,0.282,0.489]  *best*
  ep   5  loss=0.3999  acc=0.323  F1=0.287  preds=[3955,1234,2241]  f1/cls=[0.383,0.082,0.395]
  ep  10  loss=0.3204  acc=0.348  F1=0.327  preds=[2549,1757,3124]  f1/cls=[0.385,0.171,0.424]
